Now we're making an important transition.

So far:

``` text

Raw casting images
       ↓
Our custom CNN
       ↓
Learn everything from our
casting dataset
       ↓
83.22% test accuracy

```

Now we'll explore:

``` text

Large pretrained vision model
       ↓
Already learned useful
visual representations
       ↓
Adapt it to our
def_front / ok_front problem

```

This is transfer learning.

##### 1. What is transfer learning?

Suppose someone has already trained a large CNN using a huge collection of images.

During that training, the CNN learned visual representations useful for recognizing things such as:

- edges
- corners
- curves
- textures
- shapes
- complex visual patterns

Instead of starting with completely random weights:

Our custom CNN

``` text

Random weights
     ↓
Casting images
     ↓
Learn visual features
     ↓
Learn defect classification

```

we can start from previously learned weights:

Pretrained CNN

``` text

Previously learned weights
          ↓
Already knows useful
visual representations
          ↓
Our casting images
          ↓
Adapt to our task
          ↓
def_front / ok_front

```

That reuse of learned knowledge is transfer learning.

##### 2. Connection to your NLP project

You've actually already used the same general idea.

Remember MiniLM:

``` text

Support-ticket text
       ↓
Pretrained MiniLM
       ↓
384-dimensional embedding
       ↓
Our classifier
       ↓
Billing / Login /
Cancellation / Technical

```

We did not teach MiniLM language from scratch using our ~100 support tickets.

MiniLM had already learned useful language representations.

Computer vision has an analogous idea:

``` text

Casting image
       ↓
Pretrained vision network
       ↓
Learned visual representation
       ↓
Our classifier
       ↓
def_front / ok_front

```

So:

``` text

NLP                           Computer Vision

Pretrained MiniLM             Pretrained ResNet
      ↓                              ↓
Language features             Visual features
      ↓                              ↓
Our classifier                Our classifier

``` 

This is a very useful conceptual bridge.

##### 3. What does "pretrained" actually mean?


Pretrained = the model's weights were learned during an earlier training process.

Our custom CNN originally started approximately like:

``` text

Conv filters
    ↓
randomly initialized weights

Then Notebook 06 changed those weights through:

forward pass
    ↓
loss
    ↓
backpropagation
    ↓
optimizer
    ↓
updated weights

```

A pretrained model instead starts with: Already learned weights

For example, torchvision provides pretrained ResNet models.

Conceptually:

weights = ResNet18_Weights.DEFAULT

model = resnet18(
    weights=weights
)

DEFAULT means use torchvision's recommended pretrained weights for that model version.

##### 4. Where did ResNet learn these weights?

Common torchvision ResNet weights were trained on ImageNet.

ImageNet contains a very large collection of labeled natural images across many categories.

The original task might include categories unrelated to casting defects.

That naturally raises a question:

If ImageNet doesn't specifically teach our model def_front and ok_front, why would it help?

Because we're not primarily interested in transferring its original final class names.

We're transferring its learned visual representations.

##### 5. Earlier vs deeper features

Remember CNN foundations:

``` text

Early CNN layers
      ↓
simple visual patterns

Middle layers
      ↓
textures / combinations

Deeper layers
      ↓
more complex visual representations

```

A pretrained CNN has already spent substantial training learning useful filters.

For example:

``` text

ImageNet training

millions of images
      ↓
CNN learns
      ↓
edges
curves
textures
shapes
visual combinations
      ↓
trained weights

```

Those lower-level visual patterns can still be useful for a metal casting image.

The model doesn't need ImageNet to contain the exact same business problem for transfer learning to be useful.

##### 6. Our classifier currently has the wrong output

Suppose pretrained ResNet18 originally ends with something conceptually like:

ResNet visual features
        ↓
Final Linear layer
        ↓
1000 ImageNet classes

We don't want those 1000 classes.

Our problem has:

def_front
ok_front

Therefore we replace the final classifier:

``` text

PRETRAINED RESNET18

Input image
    ↓
Pretrained convolutional layers
    ↓
Learned visual features
    ↓
Original classifier
1000 outputs                 ❌

becomes:

Input image
    ↓
Pretrained convolutional layers
    ↓
Learned visual features
    ↓
New classifier
2 outputs                    ✅

[def_front, ok_front]

```
This is the core of our next implementation.

##### 7. Feature extraction vs fine-tuning

There are two important transfer-learning approaches.

- Feature extraction - We can freeze the pretrained layers:

``` text

Pretrained visual layers
        🔒 frozen
           ↓
New classifier
        trainable

```
- Frozen means:Don't update those weights during our training.

Only our new final classifier learns.

Conceptually:

for parameter in model.parameters():
    parameter.requires_grad = False

Then replace the classifier:

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes,
)

The new model.fc parameters are trainable.

- Fine-tuning

Alternatively:

``` text

Pretrained network
      ↓
allow some/all pretrained
weights to update
      ↓
adapt more specifically
to casting images

That's fine-tuning.

So:

Feature Extraction

Pretrained layers 🔒
       ↓
Only new classifier learns


Fine-Tuning

Pretrained layers 🔓
       ↓
Some/all weights continue learning

We'll experience both rather than just memorize the definitions.

##### 8. Why not always fine-tune everything?

Because fine-tuning everything:

``` text

more trainable parameters
        ↓
more computation
        ↓
potentially more training time
        ↓
greater possibility of overfitting

```
This is especially relevant to you because your current Databricks cluster is CPU-based.

Feature extraction is therefore a sensible first transfer-learning baseline.

##### 9. Pretrained preprocessing

For our custom CNN, we choose that preprocessing and trained the CNN with it.

But pretrained ResNet's weights were learned with a particular preprocessing pipeline.

Therefore we should use the preprocessing associated with those weights.

Torchvision provides it:

from torchvision.models import (
    ResNet18_Weights,
)

weights = (
    ResNet18_Weights.DEFAULT
)

preprocess = weights.transforms()

Think of it like your earlier ML work:

``` text

Train Logistic Regression
using fitted StandardScaler
        ↓
Inference
        ↓
must use compatible scaler

```
Similarly:

``` text

Pretrained ResNet
trained with expected
image preprocessing
        ↓
Our images
        ↓
use preprocessing compatible
with those weights

```

##### 10. Why transfer learning is particularly useful here

Our training source contains only: 6,633 original training images

and after creating validation:

- Train = 5,638
- Validation = 995

That's enough to learn something—as our custom CNN demonstrated—but it's relatively modest for learning visual representations completely from scratch.

Transfer learning lets us leverage prior training and then adapt those representations to our smaller domain-specific dataset.

##### 11. Important misconception to avoid

Transfer learning does not mean:

- ResNet already knows casting defects.

It means:

``` text

ResNet already learned
general visual representations
       ↓
we reuse/adapt them
       ↓
for casting defect classification

```

That's a much more accurate statement.

##### 12. What we'll compare

Eventually:

CUSTOM CNN
──────────
- Start: random weights
- Training: our casting images
- Test accuracy: 83.22%

TRANSFER LEARNING
─────────────────
- Start: pretrained weights
- Training: adapt to casting images
- Test accuracy: TBD

But remember an important scientific rule:

We shouldn't assume transfer learning will be better before measuring it.

We'll train it, evaluate it on the same untouched test dataset, and compare the results.

##### Key Learnings

1. Transfer learning reuses knowledge learned by a model during a
   previous training task.

2. A pretrained model starts with previously learned weights rather
   than completely random weights.

3. Pretrained vision models learn reusable visual representations such
   as edges, textures, shapes, and more complex patterns.

4. The original pretrained classifier can be replaced with a classifier
   appropriate for the new task.

5. Our casting problem requires two output classes:
   `def_front` and `ok_front`.

6. Feature extraction freezes pretrained layers and trains primarily
   the new classifier.

7. Fine-tuning allows some or all pretrained weights to continue
   updating for the new task.

8. Pretrained weights should be used with compatible image
   preprocessing.

9. Transfer learning can be particularly useful when the target dataset
   is much smaller than the dataset used for pretraining.

10. Transfer learning does not mean the pretrained model already knows
    casting defects. It means previously learned visual representations
    are reused and adapted to the casting-defect task.


- NLP : Text → pretrained MiniLM → learned language representation → classifier
- CV : Image → pretrained ResNet → learned visual representation → classifier